# Implement DPO Loss from Scratch

**Difficulty**: 🔴 Hard

**Companies**: Anthropic, OpenAI, DeepMind, Meta

---

### Problem Statement

Direct Preference Optimization (DPO) is a breakthrough algorithm for aligning language models with human preferences **without** needing a separate reward model or reinforcement learning. Instead of the traditional RLHF pipeline (train reward model → optimize with PPO), DPO directly optimizes the policy using preference pairs.

The key insight: the optimal policy under the RLHF objective has a closed-form relationship to the reward function, which lets us reparameterize the reward model loss as a loss over the policy directly.

The DPO loss is:

```
L_DPO = -log(sigmoid(beta * ((log_pi(chosen) - log_ref(chosen)) - (log_pi(rejected) - log_ref(rejected)))))
```

Your task is to implement DPO loss, a function to compute log probabilities of sequences under a model, and train a simple language model using DPO on synthetic preference data.

---

### Requirements

1. **dpo_loss** — Compute the DPO objective given log probabilities from policy and reference models.
2. **get_batch_logps** — Compute per-sequence log probabilities from model logits.
3. **Training** — Create a policy model, a frozen reference model, and synthetic preference pairs. Train the policy with DPO.

---

### Constraints

- ✅ The reference model must remain frozen throughout training.
- ✅ Log probabilities should be summed over tokens in each sequence (not averaged).
- ✅ After training, the policy should assign higher probability to chosen over rejected completions.
- ❌ Do **not** use any existing DPO/alignment libraries (trl, etc.).

---

<details>
  <summary>💡 Hint</summary>

  - To compute log probabilities: get logits from the model, apply log_softmax, then gather the log probs at the token indices. Sum over the sequence length dimension.
  - The DPO loss encourages the policy to increase the log-probability gap between chosen and rejected sequences, relative to the reference model.
  - Beta controls how much the policy can deviate from the reference. Typical values are 0.1-0.5.
  - Use `F.logsigmoid` for numerical stability instead of `torch.log(torch.sigmoid(...))`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

In [ ]:
# Simple language model for DPO experiments
class SimpleLM(nn.Module):
    """A minimal language model: embedding -> transformer-like layers -> logits."""
    def __init__(self, vocab_size=100, embed_dim=64, hidden_dim=128, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        layers = []
        layers.append(nn.Linear(embed_dim, hidden_dim))
        layers.append(nn.ReLU())
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
            layers.append(nn.ReLU())
        self.backbone = nn.Sequential(*layers)
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, input_ids):
        """Returns logits of shape (batch, seq_len, vocab_size)."""
        x = self.embedding(input_ids)
        x = self.backbone(x)
        return self.lm_head(x)


# Create synthetic preference data
torch.manual_seed(42)
vocab_size = 100
seq_len = 16
batch_size = 32

# "Chosen" sequences: tokens in range [0, 50) (pretend these are preferred)
# "Rejected" sequences: tokens in range [50, 100) (pretend these are dispreferred)
chosen_ids = torch.randint(0, 50, (batch_size, seq_len))
rejected_ids = torch.randint(50, 100, (batch_size, seq_len))

print(f"Chosen shape: {chosen_ids.shape}")
print(f"Rejected shape: {rejected_ids.shape}")
print(f"Chosen sample: {chosen_ids[0, :8].tolist()}")
print(f"Rejected sample: {rejected_ids[0, :8].tolist()}")

In [ ]:
def get_batch_logps(model: nn.Module, input_ids: torch.Tensor) -> torch.Tensor:
    """
    Compute the total log probability of each sequence under the model.
    
    For each sequence, compute log P(x_t | x_{<t}) for each token position
    using the model's logits, then sum over the sequence length.
    
    Args:
        model: Language model that returns logits (batch, seq_len, vocab_size).
        input_ids: Token IDs of shape (batch, seq_len).
    
    Returns:
        Tensor of shape (batch,) with total log probability per sequence.
    """
    # TODO: Get logits from model
    # TODO: Compute log probabilities with log_softmax over vocab dimension
    # TODO: Gather log probs at the actual token positions
    #       Use input_ids[:, 1:] as targets (shifted by 1 for next-token prediction)
    #       Use logits[:, :-1, :] as predictions
    # TODO: Sum log probs over sequence length to get per-sequence log probability
    ...


def dpo_loss(policy_chosen_logps: torch.Tensor,
             policy_rejected_logps: torch.Tensor,
             ref_chosen_logps: torch.Tensor,
             ref_rejected_logps: torch.Tensor,
             beta: float = 0.1) -> torch.Tensor:
    """
    Compute the DPO loss.
    
    L = -log(sigmoid(beta * ((log_pi_chosen - log_ref_chosen) - (log_pi_rejected - log_ref_rejected))))
    
    Args:
        policy_chosen_logps: Log probs of chosen sequences under policy. Shape (batch,).
        policy_rejected_logps: Log probs of rejected sequences under policy. Shape (batch,).
        ref_chosen_logps: Log probs of chosen sequences under reference. Shape (batch,).
        ref_rejected_logps: Log probs of rejected sequences under reference. Shape (batch,).
        beta: Temperature parameter controlling deviation from reference.
    
    Returns:
        Scalar DPO loss (mean over batch).
    """
    # TODO: Compute log ratios for chosen: log_pi_chosen - log_ref_chosen
    # TODO: Compute log ratios for rejected: log_pi_rejected - log_ref_rejected
    # TODO: Compute DPO loss: -mean(logsigmoid(beta * (chosen_ratio - rejected_ratio)))
    ...

In [ ]:
# === Validation ===
torch.manual_seed(42)

# Create policy and reference models
policy_model = SimpleLM(vocab_size=vocab_size)
ref_model = copy.deepcopy(policy_model)
for param in ref_model.parameters():
    param.requires_grad = False

# Compute reference log probs (fixed throughout training)
ref_model.eval()
with torch.no_grad():
    ref_chosen_logps = get_batch_logps(ref_model, chosen_ids)
    ref_rejected_logps = get_batch_logps(ref_model, rejected_ids)

# Training loop
print("=== DPO Training ===")
optimizer = torch.optim.Adam(policy_model.parameters(), lr=1e-4)
losses = []

for epoch in range(100):
    policy_model.train()
    optimizer.zero_grad()

    policy_chosen_logps = get_batch_logps(policy_model, chosen_ids)
    policy_rejected_logps = get_batch_logps(policy_model, rejected_ids)

    loss = dpo_loss(policy_chosen_logps, policy_rejected_logps,
                    ref_chosen_logps, ref_rejected_logps, beta=0.1)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f}")

print(f"\nInitial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")

# Test 1: Loss should decrease
print("\n=== Loss Decrease Test ===")
assert losses[-1] < losses[0], "DPO loss should decrease during training!"
print("PASSED\n")

# Test 2: Policy should prefer chosen over rejected
print("=== Preference Test ===")
policy_model.eval()
with torch.no_grad():
    final_chosen_logps = get_batch_logps(policy_model, chosen_ids)
    final_rejected_logps = get_batch_logps(policy_model, rejected_ids)

chosen_preferred = (final_chosen_logps > final_rejected_logps).float().mean().item()
print(f"Fraction where policy prefers chosen: {chosen_preferred:.2%}")
assert chosen_preferred > 0.5, "Policy should prefer chosen over rejected completions!"
print("PASSED\n")

# Test 3: Policy should diverge from reference on preference data
print("=== Divergence Test ===")
with torch.no_grad():
    chosen_ratio = (final_chosen_logps - ref_chosen_logps).mean().item()
    rejected_ratio = (final_rejected_logps - ref_rejected_logps).mean().item()
print(f"Chosen log-ratio (policy/ref):   {chosen_ratio:.4f}")
print(f"Rejected log-ratio (policy/ref): {rejected_ratio:.4f}")
assert chosen_ratio > rejected_ratio, "Policy should increase chosen probability more than rejected!"
print("PASSED\n")

print("All tests passed!")